# CaST-POI v2 — canonical **per-user LOO** + full-ranking (code5-loo)

Uses the **exact same LOO split as the baselines**: `build_loo_split.py` (from
`kdd_baselines/`) re-splits `data_official` into per-user leave-one-out (last=test,
2nd-last=val, rest=train) **with unseen-POI removal**, writing `data_loo/` that the
harness reads unchanged. CaST-POI and the RecBole baselines therefore run on one
**bit-identical** split (verified: same test samples, same order, same fingerprint).

Full-vocabulary ranking, |POI|=4980/7832/9689. Open in 3 Colabs, SEED=42/43/44.
Prereq: `data_official` on Drive at `/content/drive/MyDrive/castpoi/data_official/`. GPU.

### 1. GPU + Drive + data_official

In [1]:
# 1) GPU + Drive + stage data_official
import os, torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '(no GPU!)')
from google.colab import drive; drive.mount('/content/drive')
DRIVE_ROOT='/content/drive/MyDrive/castpoi'
WORK='/content/work'; os.makedirs(WORK,exist_ok=True); os.chdir(WORK)
assert os.path.isdir(f'{DRIVE_ROOT}/data_official'), f'no data_official at {DRIVE_ROOT}'
os.makedirs(f'{WORK}/data_official',exist_ok=True)
!cp -rn {DRIVE_ROOT}/data_official/* {WORK}/data_official/ 2>/dev/null || cp -r {DRIVE_ROOT}/data_official/* {WORK}/data_official/
os.makedirs('castpoi',exist_ok=True)
os.environ['CASTPOI_EXTRA_PATH']=WORK
os.environ['OFFICIAL_DIR']=f'{WORK}/data_loo'     # <-- LOO dir (built in cell 4)
os.environ['CASTPOI_RESULTS']=f'{DRIVE_ROOT}/castpoi_loo_results'
print('data_official:', sorted(os.listdir(f'{WORK}/data_official')))

CUDA: True NVIDIA L4
Mounted at /content/drive
data_official: ['ca', 'nyc', 'tky']


### 1b. Seed

In [2]:
# 1b) *** SEED for THIS Colab *** (42 / 43 / 44)
SEED = 44
print('seed', SEED)

seed 44


### 2. Write castpoi harness

In [3]:
%%writefile castpoi/__init__.py
"""CaST-POI."""


Writing castpoi/__init__.py


In [4]:
%%writefile castpoi/utils.py
"""Seeding, environment capture, logging, and artifact IO.

Every run records enough environment metadata to prove where and when it ran.
"""
import json
import logging
import os
import platform
import random
import socket
import subprocess
import sys
import time
from pathlib import Path
from typing import Any, Dict, Optional

import numpy as np
import torch


def set_seed(seed: int, deterministic: bool = True) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def pick_device(requested: str = "auto") -> torch.device:
    """auto -> cuda if present, else cpu.

    MPS is deliberately NOT chosen automatically. It is available on Apple
    silicon and this model trains to a NaN loss on it while CPU and CUDA train
    normally; the cause has not been isolated. Since a NaN loss used to surface
    as a perfect 100% score, an automatic device that quietly breaks training is
    the last thing this package should do. Pass --device mps explicitly if you
    want to investigate it.
    """
    if requested != "auto":
        return torch.device(requested)
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def _git_commit() -> Optional[str]:
    try:
        out = subprocess.run(
            ["git", "rev-parse", "HEAD"],
            cwd=Path(__file__).resolve().parent,
            capture_output=True, text=True, timeout=5,
        )
        return out.stdout.strip() or None
    except Exception:
        return None


def env_info() -> Dict[str, Any]:
    """Provenance block embedded in every result file."""
    info = {
        "hostname": socket.gethostname(),
        "platform": platform.platform(),
        "python": sys.version.split()[0],
        "torch": torch.__version__,
        "numpy": np.__version__,
        "cuda_available": torch.cuda.is_available(),
        "git_commit": _git_commit(),
        "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "argv": sys.argv,
        "slurm_job_id": os.environ.get("SLURM_JOB_ID"),
    }
    if torch.cuda.is_available():
        info["gpu_name"] = torch.cuda.get_device_name(0)
        info["gpu_count"] = torch.cuda.device_count()
    return info


def setup_logger(log_path: Path, name: str = "castpoi") -> logging.Logger:
    log_path.parent.mkdir(parents=True, exist_ok=True)
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    logger.propagate = False
    fmt = logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%Y-%m-%d %H:%M:%S")

    fh = logging.FileHandler(log_path, mode="a")
    fh.setFormatter(fmt)
    logger.addHandler(fh)

    sh = logging.StreamHandler(sys.stdout)
    sh.setFormatter(fmt)
    logger.addHandler(sh)
    return logger


class _Encoder(json.JSONEncoder):
    def default(self, o):
        if isinstance(o, (np.integer,)):
            return int(o)
        if isinstance(o, (np.floating,)):
            return float(o)
        if isinstance(o, np.ndarray):
            return o.tolist()
        if isinstance(o, Path):
            return str(o)
        return super().default(o)


def write_json(path: Path, obj: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=2, cls=_Encoder)
    tmp.replace(path)  # atomic: a killed job never leaves a half-written result


def read_json(path: Path) -> Any:
    with open(path) as f:
        return json.load(f)


def append_jsonl(path: Path, obj: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a") as f:
        f.write(json.dumps(obj, cls=_Encoder) + "\n")


def count_params(model: torch.nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


Writing castpoi/utils.py


In [5]:
%%writefile castpoi/timeparse.py
"""Time handling for the LLM4POI preprocessed files.

`UTCTimeOffsetEpoch` is unusable: it is `UTCTimeOffset` passed through a naive
`datetime.timestamp()` on a UTC+10/+11 machine, so the same offset appears for
New York, Tokyo and California alike, and hour-of-day read from it places Tokyo's
quietest hour at 17:00.

`UTCTimeOffset` is used instead. Its meaning differs per dataset:
  nyc, tky : local wall clock, offset applied (Foursquare).
  ca       : raw UTC, offset not applied (Gowalla).

`assert_human_rhythm` checks the result: under the correct reading each city
shows a 3-5am trough and a daytime peak.

Two quantities come out of this module and are not interchangeable:
  ts_utc       absolute instant, for elapsed time between check-ins.
  hour / dow   local calendar position, for the periodic features.
"""
from typing import Dict, Tuple

import numpy as np
import pandas as pd

# tz            IANA zone of the city.
# column_means  what the `UTCTimeOffset` column actually holds.
TIME_SPEC: Dict[str, Dict[str, str]] = {
    "nyc": {"tz": "America/New_York", "column_means": "local"},
    "tky": {"tz": "Asia/Tokyo", "column_means": "local"},
    "ca": {"tz": "America/Los_Angeles", "column_means": "utc"},
}


class TimeParseError(RuntimeError):
    pass


def parse_times(df: pd.DataFrame, dataset: str) -> pd.DataFrame:
    """Add ts_utc (int seconds), local_hour (float 0-24), local_dow (0=Mon).

    `UTCTimeOffsetEpoch` is ignored entirely. It is never read by this package.
    """
    ds = dataset.lower()
    if ds not in TIME_SPEC:
        raise TimeParseError(f"no time spec for dataset {ds!r}")
    spec = TIME_SPEC[ds]

    if "UTCTimeOffset" not in df.columns:
        raise TimeParseError(
            f"{ds}: column 'UTCTimeOffset' is missing. The corrupt "
            f"'UTCTimeOffsetEpoch' column is not an acceptable substitute.")

    naive = pd.to_datetime(df["UTCTimeOffset"])
    if naive.isna().any():
        raise TimeParseError(f"{ds}: {naive.isna().sum()} unparseable timestamps")

    if spec["column_means"] == "local":
        # Wall clock in the city. Localize to recover the instant. DST fall-back
        # hours are ambiguous; resolve to standard time and shift nonexistent
        # spring-forward times rather than dropping check-ins.
        aware_local = naive.dt.tz_localize(spec["tz"], ambiguous=False, nonexistent="shift_forward")
        utc = aware_local.dt.tz_convert("UTC")
        local = aware_local
    else:
        utc = naive.dt.tz_localize("UTC")
        local = utc.dt.tz_convert(spec["tz"])

    out = df.copy()
    out["ts_utc"] = (utc.astype("int64") // 10 ** 9).astype("int64")
    out["local_hour"] = (local.dt.hour + local.dt.minute / 60.0 + local.dt.second / 3600.0).astype("float32")
    out["local_dow"] = local.dt.dayofweek.astype("int8")  # 0=Monday
    return out


def rhythm_stats(local_hour: np.ndarray) -> Dict[str, float]:
    """Shape of the daily check-in rhythm."""
    h = np.asarray(local_hour)
    dist = np.bincount(h.astype(int).clip(0, 23), minlength=24) / max(len(h), 1)
    night = float(dist[3:6].sum())
    day = float(dist[11:22].sum())
    return {
        "night_3_6_frac": night,
        "day_11_22_frac": day,
        "day_night_ratio": day / max(night, 1e-9),
        "argmin_hour": int(dist.argmin()),
        "argmax_hour": int(dist.argmax()),
        "hist": dist.tolist(),
    }


def assert_human_rhythm(local_hour: np.ndarray, dataset: str, min_ratio: float = 3.0) -> Dict[str, float]:
    """Raise if the parsed local time is not a plausible human rhythm.

    A guard, not a formality. Every wrong reading of these columns that we found
    puts the daily minimum somewhere between 11:00 and 17:00. Real check-ins
    trough between 03:00 and 06:00.
    """
    st = rhythm_stats(local_hour)
    bad_min = not (1 <= st["argmin_hour"] <= 7)
    bad_ratio = st["day_night_ratio"] < min_ratio
    if bad_min or bad_ratio:
        raise TimeParseError(
            f"{dataset}: parsed local time does not look like human behaviour "
            f"(quietest hour = {st['argmin_hour']:02d}:00, busiest = {st['argmax_hour']:02d}:00, "
            f"day/night ratio = {st['day_night_ratio']:.1f}). Expected the trough between "
            f"03:00 and 06:00. The time column is being read wrong; check TIME_SPEC[{dataset!r}].")
    return st


Writing castpoi/timeparse.py


In [6]:
%%writefile castpoi/config.py
"""Model and training configuration.

One BASE_CONFIG is used for every dataset; there are no per-dataset defaults.
Preprocessing and splitting are not configurable here. They come from the
official files loaded by official.py and from build_loo_split.py; see DATA.md.
"""
import copy
from typing import Any, Dict

BASE_CONFIG: Dict[str, Any] = {
    # Model
    "poi_embed_dim": 128,
    "slot_embed_dim": 16,
    "spatial_dim": 32,
    "dist_embed_dim": 16,
    "num_dist_buckets": 8,
    # Revisit gate: MLP width, and the window the visit counts are taken over.
    # The sequence encoder still sees only max_history_len; counting is cheap,
    # so this window can be longer.
    "repeat_gate_hidden": 32,
    "repeat_history_len": 512,

    # Training objective. "sampled" scores the positive against `num_negatives`
    # sampled negatives; "full" scores it against the entire POI vocabulary.
    # Evaluation is always full-vocabulary, and the RecBole baselines train with
    # full-vocabulary cross-entropy, so run.py sets "full". The vocabulary is
    # small enough (4,980 / 7,832 / 9,689) for a dense softmax.
    "train_objective": "sampled",   # "sampled" | "full"
    "num_negatives": 499,           # ignored when train_objective == "full"
    "batch_size": 512,
    "eval_batch_size": 1024,
    "num_epochs": 50,
    "learning_rate": 2e-3,
    "weight_decay": 1e-4,
    "dropout": 0.1,
    "gradient_clip": 5.0,
    "early_stopping_patience": 10,
    "warmup_epochs": 3,
    "label_smoothing": 0.02,
    "explore_weight": 1.5,
    "bpr_weight": 0.5,
    "bpr_margin": 1.0,

    # Model input truncation, not a data filter.
    "max_history_len": 50,
    # Selects the validation metric. run.py widens this to [1, 5, 10, 20] so the
    # saved metrics cover every K the tables use.
    "eval_ks": [5, 10],
}

DATASETS = ("nyc", "tky", "ca")


def resolve_config(dataset: str, overrides: Dict[str, Any] = None) -> Dict[str, Any]:
    """Effective config for `dataset`. Identical for every dataset by default."""
    ds = dataset.lower()
    if ds not in DATASETS:
        raise ValueError(f"unknown dataset {dataset!r}; choose from {list(DATASETS)}")
    cfg = copy.deepcopy(BASE_CONFIG)
    if overrides:
        cfg.update({k: v for k, v in overrides.items() if v is not None})
    cfg["dataset"] = ds

    for gone in ("split_protocol", "test_size", "min_poi_checkins", "min_user_checkins"):
        if gone in cfg:
            raise ValueError(
                f"{gone!r} is not a configuration option: preprocessing comes from "
                f"the official STHGCN/LLM4POI output, not from this package. "
                f"See DATA.md.")
    return cfg


Writing castpoi/config.py


In [7]:
%%writefile castpoi/metrics.py
"""Full-vocabulary ranking metrics.

Metrics are returned per sample, so the reported mean and any paired test are
computed from the same array.
"""
from typing import Dict, List, Sequence

import numpy as np

# K values written to metrics.json, independent of `eval_ks`, which only selects
# the validation metric.
REPORT_KS = (1, 5, 10, 20)
import torch


def per_sample_ranks(scores: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    """1-indexed rank of the target under full-vocabulary scoring.

    Ties use the mid-rank convention

        rank = 1 + #{strictly higher} + #{tied} / 2

    rather than 1 + #{strictly higher}. The optimistic rule is only equivalent
    when scores are dense: a count-based scorer assigns exactly 0 to every
    unvisited POI, so an unvisited target ties with thousands of items and would
    be credited with a near-top rank. Dense neural scorers are unaffected, so the
    optimistic rule would favour whichever method produces more ties.

    Non-finite scores raise instead of ranking, because every comparison with NaN
    is False and a diverged model would otherwise be scored as rank 1 everywhere,
    i.e. a perfect result.
    """
    if not torch.isfinite(scores).all():
        n_bad = int((~torch.isfinite(scores)).sum())
        raise ValueError(
            f"{n_bad} of {scores.numel()} scores are NaN or Inf. Ranking them would "
            f"report rank 1 for every sample (every comparison with NaN is False), "
            f"i.e. a perfect 100% score from a broken model. Check for a diverged "
            f"loss or an unsupported device dtype.")
    tgt = scores.gather(1, targets.unsqueeze(1))
    greater = (scores > tgt).sum(1)
    tied = (scores == tgt).sum(1) - 1          # exclude the target itself
    return greater + 1 + tied.float() / 2.0


def metrics_from_ranks(ranks: np.ndarray, ks: Sequence[int] = (5, 10, 20)) -> Dict[str, np.ndarray]:
    """Per-sample metric vectors. Mean of each vector is the reported number."""
    out: Dict[str, np.ndarray] = {}
    for k in ks:
        out[f"HR@{k}"] = (ranks <= k).astype(np.float64)
        out[f"NDCG@{k}"] = np.where(ranks <= k, 1.0 / np.log2(ranks + 1.0), 0.0)
    out["MRR"] = 1.0 / ranks
    return out


def summarize(per_sample: Dict[str, np.ndarray]) -> Dict[str, float]:
    return {k: float(v.mean()) for k, v in per_sample.items()}


def format_metrics(m: Dict[str, float], pct: bool = True) -> str:
    order = sorted(m.keys(), key=lambda s: (s.split("@")[0], int(s.split("@")[1]) if "@" in s else 0))
    return " | ".join(f"{k}: {m[k] * 100:.2f}" if pct else f"{k}: {m[k]:.4f}" for k in order)




Writing castpoi/metrics.py


In [8]:
%%writefile castpoi/official.py
"""Load the official STHGCN/LLM4POI splits.

`data_official/<ds>/` holds the output of LLM4POI's own preprocessing, run
unmodified; see DATA.md. The regenerated `train_sample.csv` is byte-identical to
the published `w11wo/LLM4POI` files for all three datasets, apart from 424
mojibake rows in that upload.

The properties below come from the official pipeline, not from this module:
min_poi_freq / min_user_freq of 9 / 9 with `count > freq` semantics (CA is
filtered twice; NYC arrives pre-split from GETNext), a global chronological
80/10/10 split over all check-ins, 24-hour session gaps with singleton sessions
dropped, and removal of val/test rows whose user or POI is absent from train.

Two things are done here because the official files leave them:

1. `UTCTimeOffsetEpoch` is corrupt in every published file (see timeparse.py), so
   `UTCTimeOffset` is parsed instead.
2. POI id 0 is a real POI in their label encoding, while the model reserves 0 for
   padding, so ids are shifted where needed.
"""
import hashlib
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd

from .timeparse import TIME_SPEC, assert_human_rhythm, parse_times

DEFAULT_OFFICIAL = Path(__file__).resolve().parent.parent / "data_official"

FILES = {
    "train": "train_sample.csv",
    "val": "validate_sample_with_traj.csv",
    "test": "test_sample_with_traj.csv",
}

SOURCE = {
    "nyc": {"note": "Foursquare New York City", "platform": "Foursquare",
            "provenance": "GETNext pre-split NYC_{train,val,test}.csv"},
    "tky": {"note": "Foursquare Tokyo", "platform": "Foursquare",
            "provenance": "STHGCN filter(9,9) + global chronological 80/10/10"},
    "ca": {"note": "Gowalla California", "platform": "Gowalla",
           "provenance": "STHGCN filter(9,9) applied TWICE + global chronological 80/10/10"},
}

# The official pipeline label-encodes ids under two conventions: NYC yields
# ids 0..N-1 with padding N, while TKY and CA yield ids 1..N with padding 0. So
# PoiId==0 is a real POI in NYC and the padding bucket in TKY/CA. Treating them
# alike would count the padding bucket as an entity and, given that 0 is reserved
# for sequence padding here, would leave a dead embedding row in TKY/CA. The
# convention is declared explicitly rather than inferred from min(), so a split
# that happens not to contain id 0 cannot be misread.
ID_CONVENTION = {"nyc": "zero_indexed", "tky": "one_indexed", "ca": "one_indexed"}


class OfficialDataMissing(RuntimeError):
    pass


def _read(ds: str, root: Path) -> Dict[str, pd.DataFrame]:
    d = root / ds
    out = {}
    for split, fn in FILES.items():
        p = d / fn
        if not p.exists():
            raise OfficialDataMissing(
                f"{p} not found. Regenerate with LLM4POI's own pipeline; see "
                f"DATA.md. This code will not "
                f"substitute its own preprocessing.")
        out[split] = pd.read_csv(p, low_memory=False)
    return out


def poi_shift(ds: str) -> int:
    """Offset mapping the official PoiId onto our vocabulary, where 0 = padding.

    zero_indexed (nyc): official ids 0..N-1 -> 1..N, shift +1.
    one_indexed  (tky, ca): official ids are already 1..N with 0 as their own
                            padding bucket -> no shift.
    """
    return 1 if ID_CONVENTION[ds] == "zero_indexed" else 0


def real_id_range(ds: str, train: pd.DataFrame, col: str) -> Tuple[int, int]:
    """(n_distinct_real, our_max_index) for a label-encoded column."""
    if ID_CONVENTION[ds] == "zero_indexed":
        n = int(train[col].max()) + 1        # 0..max are all real; padding is max+1
    else:
        n = int(train[col].max())            # 1..max are real; 0 is padding
    return n, n


def _trajectories(df: pd.DataFrame, ds: str) -> Dict[int, List[Dict]]:
    df = parse_times(df, ds)
    # mergesort = stable, so exact (UserId, ts_utc) ties keep their file order
    # instead of pandas' quicksort tie-break. TKY ships 467 exact duplicate
    # (UserId, ts_utc, PoiId) rows and CA 35; with an unstable sort, a duplicate
    # of the target could land in the *history* of its own eval sample. Only 36
    # samples out of 60k were affected, so this changes no conclusion, but it is
    # the difference between "causal" and "causal except when pandas feels like it".
    df = df.sort_values(["UserId", "ts_utc"], kind="mergesort")
    has_cat = "PoiCategoryName" in df.columns
    # check_ins_id rides along unused by the model. It is the only stable join key
    # back to a foreign implementation's test set: the official pipeline assigns it
    # (preprocess_main.py:45) as a rank over UTCTimeOffset, a wall-clock string, so
    # it survives the timezone corruption that makes UTCTimeOffsetEpoch unusable
    # across machines. Without it, comparing against a re-run of STHGCN means
    # replaying this function's grouping from outside and hoping it stays in sync.
    cols = ["UserId", "PoiId", "ts_utc", "local_hour", "local_dow",
            "Latitude", "Longitude", "pseudo_session_trajectory_id", "check_ins_id"]
    if has_cat:
        cols.append("PoiCategoryName")

    shift = poi_shift(ds)
    trajs: Dict[int, List[Dict]] = {}
    for row in df[cols].itertuples(index=False, name=None):
        trajs.setdefault(int(row[0]), []).append({
            "poi_idx": int(row[1]) + shift,      # 0 reserved for padding
            "ts_utc": float(row[2]),
            "hour": float(row[3]),
            "dow": int(row[4]),
            "latitude": float(row[5]),
            "longitude": float(row[6]),
            "traj_id": int(row[7]),
            "check_ins_id": int(row[8]),
            "category": row[9] if has_cat else "Unknown",
        })
    return trajs


def fingerprint(splits: Dict[str, Dict], num_pois: int) -> str:
    h = hashlib.sha256()
    h.update(f"OFFICIAL_V1|{num_pois}".encode())
    for name in ("train", "val", "test"):
        h.update(f"|{name}|".encode())
        for uid in sorted(splits[name]):
            h.update(f"{uid}:".encode())
            h.update(np.array([c["poi_idx"] for c in splits[name][uid]], dtype=np.int64).tobytes())
            h.update(np.array([c["ts_utc"] for c in splits[name][uid]], dtype=np.int64).tobytes())
    return h.hexdigest()[:16]


def load_official(ds: str, root: Path = None) -> Dict:
    ds = ds.lower()
    root = Path(root or DEFAULT_OFFICIAL)
    info = SOURCE[ds]
    print(f"\n{'=' * 68}\n[official] {ds.upper()} ({info['note']}, {info['platform']})\n"
          f"[official] provenance: {info['provenance']}\n{'=' * 68}")

    raw = _read(ds, root)
    for k, df in raw.items():
        print(f"[official] {k:5s}: {len(df):>7,} check-ins  "
              f"({(root / ds / FILES[k]).stat().st_size / 1e6:.1f} MB)")

    # Vocabulary comes from TRAIN, exactly as their id_encode does. val/test have
    # already had unseen users and POIs removed upstream.
    shift = poi_shift(ds)
    n_train_pois, max_idx = real_id_range(ds, raw["train"], "PoiId")
    n_users, _ = real_id_range(ds, raw["train"], "UserId")
    num_pois = max_idx + 1                           # index 0 is our padding

    for k in ("val", "test"):
        lo, hi = raw[k]["PoiId"].min() + shift, raw[k]["PoiId"].max() + shift
        assert 1 <= lo and hi <= max_idx, (
            f"{ds}/{k} PoiId maps outside 1..{max_idx} (got {lo}..{hi}); "
            f"unseen POIs should have been removed upstream")

    splits = {k: _trajectories(df, ds) for k, df in raw.items()}
    used = {c["poi_idx"] for s in splits.values() for t in s.values() for c in t}
    assert 0 not in used, f"{ds}: padding index 0 leaked into the data"
    dead = set(range(1, num_pois)) - used
    if dead:
        print(f"[official] note: {len(dead)} vocabulary slots never appear in any split")

    all_df = parse_times(pd.concat(raw.values()), ds)
    st = assert_human_rhythm(all_df["local_hour"].values, ds)

    # POI metadata is aggregated over the train split only, so no statistic is
    # computed over test. POIs absent from train keep coordinate (0,0) and
    # category Unknown; their embeddings never receive a gradient.
    train_df = all_df[all_df["_split"] == "train"] if "_split" in all_df.columns else \
        parse_times(raw["train"], ds)
    poi_locations = np.zeros((num_pois, 2), dtype=np.float64)
    poi_categories: Dict[int, str] = {}
    agg = {"Latitude": "mean", "Longitude": "mean"}
    if "PoiCategoryName" in train_df.columns:
        agg["PoiCategoryName"] = lambda x: x.mode().iloc[0] if len(x.mode()) else "Unknown"
    for pid, row in train_df.groupby("PoiId").agg(agg).iterrows():
        i = int(pid) + shift
        if not 1 <= i < num_pois:
            continue                                 # the official padding bucket
        poi_locations[i] = [row["Latitude"], row["Longitude"]]
        poi_categories[i] = row.get("PoiCategoryName", "Unknown")

    counts = np.zeros(num_pois, dtype=np.float64)
    for traj in splits["train"].values():
        for c in traj:
            counts[c["poi_idx"]] += 1
    counts[0] = 0.0
    popularity = counts / counts.sum()

    n_check = sum(len(df) for df in raw.values())
    n_traj = int(pd.concat(raw.values())["pseudo_session_trajectory_id"].nunique())
    stats = {
        "dataset": ds.upper(), "platform": info["platform"], "note": info["note"],
        "provenance": info["provenance"], "id_convention": ID_CONVENTION[ds],
        "users": n_users, "pois": n_train_pois, "checkins": n_check,
        "trajectories": n_traj,
        "sparsity_pct": 100 * (1 - n_check / (n_users * n_train_pois)),
        "avg_traj_len": n_check / n_traj,
        "train_checkins": len(raw["train"]), "val_checkins": len(raw["val"]),
        "test_checkins": len(raw["test"]),
        "train_users": len(splits["train"]), "test_users": len(splits["test"]),
        "tz": TIME_SPEC[ds]["tz"], "time_column_means": TIME_SPEC[ds]["column_means"],
        "rhythm_trough_hour": st["argmin_hour"], "rhythm_day_night_ratio": st["day_night_ratio"],
        "num_pois_incl_pad": num_pois,
        "data_fingerprint": fingerprint(splits, num_pois),
    }
    print(f"[official] users={n_users:,} POIs={n_train_pois:,} check-ins={n_check:,} "
          f"trajectories={n_traj:,}")
    print(f"[official] local time OK (trough {st['argmin_hour']:02d}:00, "
          f"day/night {st['day_night_ratio']:.1f}) via {TIME_SPEC[ds]['tz']}, "
          f"column read as {TIME_SPEC[ds]['column_means']}")
    print(f"[official] fingerprint {stats['data_fingerprint']}")

    # Time origin. The model only ever uses DIFFERENCES of timestamps, so a
    # constant offset cancels exactly; measuring hours from the dataset's own
    # start instead of from 1970 keeps the magnitude near 1e4 rather than 1e9.
    # That is what lets the tensors be float32 without reintroducing the 128 s
    # quantization bug (float32 ulp at 1.3e9 s is 128 s; at 1.3e4 h it is 3.5 s).
    # It also makes the code run on MPS, which cannot hold float64 at all.
    t_ref = min(c["ts_utc"] for s_ in splits.values() for t in s_.values() for c in t)

    return {
        "t_ref": float(t_ref),
        "train_data": splits["train"], "val_data": splits["val"], "test_data": splits["test"],
        "poi_locations": poi_locations, "poi_categories": poi_categories,
        "poi_popularity": popularity, "num_pois": num_pois, "stats": stats,
        "default_lat": float(train_df["Latitude"].mean()),
        "default_lon": float(train_df["Longitude"].mean()),
    }


Writing castpoi/official.py


In [9]:
%%writefile castpoi/data.py
"""Torch datasets and dataloaders.

No preprocessing happens here. Filtering and splitting come from the official
files loaded by official.py and from build_loo_split.py; see DATA.md. This module
only turns those trajectories into batched tensors.
"""
import math
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset


class DataUnavailable(RuntimeError):
    """Kept for API compatibility. Real data errors now raise OfficialDataMissing."""


def _pack_repeat_history(history: List[Dict], max_len: int) -> List[int]:
    """POI ids over a long window, used only by the revisit features.

    Kept separate from the sequence window: attention is O(L) per candidate, so
    the reader truncates to max_history_len, but counting visits is cheap and a
    longer window classifies more targets correctly as revisits.
    """
    return [c["poi_idx"] for c in history[-max_len:]]


def _pack(history: List[Dict], max_len: int, default_lat: float, default_lon: float,
          t_ref: float = 0.0):
    """Truncate to the last max_len, then LEFT-pad to a fixed width.

    Timestamps come out as HOURS SINCE t_ref, not raw epoch seconds. See
    official.load_official for why: differences are all the model uses, and the
    smaller magnitude is what makes float32 safe (and MPS possible)."""
    h = history[-max_len:]
    n = len(h)
    pad = max_len - n
    poi = [0] * pad + [c["poi_idx"] for c in h]
    ts = [(h[0]["ts_utc"] - t_ref) / 3600.0] * pad + [(c["ts_utc"] - t_ref) / 3600.0 for c in h]
    hour = [0.0] * pad + [c["hour"] for c in h]
    dow = [0] * pad + [c["dow"] for c in h]
    loc = [[default_lat, default_lon]] * pad + [[c["latitude"], c["longitude"]] for c in h]
    return poi, ts, hour, dow, loc, n


def _tensors(poi, ts, hour, dow, loc, n, target, t_ref=0.0, rep_hist=None, rep_len=0):
    d = {
        "poi_ids": torch.tensor(poi, dtype=torch.long),
        "ts_hours": torch.tensor(ts, dtype=torch.float32),   # hours since t_ref
        "hour": torch.tensor(hour, dtype=torch.float32),
        "dow": torch.tensor(dow, dtype=torch.long),
        "locations": torch.tensor(loc, dtype=torch.float32),
        "seq_len": torch.tensor(n, dtype=torch.long),
        "target_poi": torch.tensor(target["poi_idx"], dtype=torch.long),
        "target_hour": torch.tensor(target["hour"], dtype=torch.float32),
        "target_dow": torch.tensor(target["dow"], dtype=torch.long),
        "target_location": torch.tensor([target["latitude"], target["longitude"]], dtype=torch.float32),
    }
    if rep_hist is not None:
        pad = rep_len - len(rep_hist)
        d["repeat_hist"] = torch.tensor([0] * pad + rep_hist, dtype=torch.long)
    return d


class POITrainDataset(Dataset):
    def __init__(self, train_data, num_pois, poi_popularity, config,
                 default_lat=0.0, default_lon=0.0, t_ref=0.0):
        self.num_pois = num_pois
        self.t_ref = t_ref
        # Under train_objective="full" nothing consumes neg_ids, and sampling
        # them anyway would cost ~200 ms per batch of 512 for a tensor the
        # training step throws away.
        self.sample_negatives = config.get("train_objective", "sampled") != "full"
        self.num_negatives = config["num_negatives"]
        self.max_history_len = config["max_history_len"]
        self.repeat_history_len = config.get("repeat_history_len", 512)
        self.default_lat, self.default_lon = default_lat, default_lon
        self.valid_idx = np.where(poi_popularity > 0)[0]
        p = poi_popularity[self.valid_idx]
        self.valid_probs = p / p.sum()
        # np.random.choice(p=...) rebuilds an O(|V|) cumulative sum on EVERY call.
        # At 499 negatives x ~82k samples per epoch that dominated the whole
        # training step: measured 38 s/epoch of pure data loading on a fast CPU,
        # which on a 2-vCPU Colab runtime left the GPU idle ~75% of the time.
        # Build the CDF once; sample with searchsorted, same distribution.
        self._cdf = np.cumsum(self.valid_probs)
        self._cdf[-1] = 1.0

        self.samples = []
        for uid, traj in train_data.items():
            seen = set()
            for i in range(1, len(traj)):
                seen.add(traj[i - 1]["poi_idx"])
                self.samples.append({"history": traj[:i], "target": traj[i],
                                     "is_explore": traj[i]["poi_idx"] not in seen})

    def __len__(self):
        return len(self.samples)

    def _sample_popularity(self, size: int) -> np.ndarray:
        """Popularity-weighted draw via the prebuilt CDF. Same distribution as
        np.random.choice(p=self.valid_probs), without its per-call O(|V|) setup."""
        return self.valid_idx[np.searchsorted(self._cdf, np.random.random(size))]

    def _sample_negatives(self, target: int) -> List[int]:
        """Half popularity-weighted, half uniform, de-duplicated, target excluded.

        Plain Python with a set: at these sizes (~500 draws over a 5k vocabulary)
        a vectorised numpy version measured slower. Costs roughly 200 ms per batch
        of 512, which num_workers > 0 hides.
        """
        n_pop = self.num_negatives // 2
        neg = set()
        for _ in range(8):
            if len(neg) >= n_pop:
                break
            neg.update(int(c) for c in self._sample_popularity(n_pop * 2) if c != target)
            if len(neg) > n_pop:
                neg = set(list(neg)[:n_pop])
        for _ in range(8):
            if len(neg) >= self.num_negatives:
                break
            for c in np.random.randint(1, self.num_pois, size=(self.num_negatives - len(neg)) * 2):
                if c != target:
                    neg.add(int(c))
                if len(neg) >= self.num_negatives:
                    break
        out = list(neg)[: self.num_negatives]
        while len(out) < self.num_negatives:
            r = int(np.random.randint(1, self.num_pois))
            if r != target:
                out.append(r)
        return out

    def __getitem__(self, idx):
        s = self.samples[idx]
        packed = _pack(s["history"], self.max_history_len, self.default_lat,
                       self.default_lon, self.t_ref)
        d = _tensors(*packed, s["target"], self.t_ref,
                     _pack_repeat_history(s["history"], self.repeat_history_len),
                     self.repeat_history_len)
        if self.sample_negatives:
            d["neg_ids"] = torch.tensor(self._sample_negatives(s["target"]["poi_idx"]), dtype=torch.long)
        d["is_explore"] = torch.tensor(float(s["is_explore"]), dtype=torch.float32)
        return d


class POIEvalDataset(Dataset):
    def __init__(self, history_base: Dict, eval_data: Dict, config,
                 default_lat=0.0, default_lon=0.0, t_ref=0.0):
        self.max_history_len = config["max_history_len"]
        self.repeat_history_len = config.get("repeat_history_len", 512)
        self.t_ref = t_ref
        self.default_lat, self.default_lon = default_lat, default_lon
        self.samples = []
        for uid, traj in eval_data.items():
            base = history_base.get(uid, [])
            for i in range(len(traj)):
                history = base + traj[:i]
                if history:
                    self.samples.append({"history": history, "target": traj[i]})

    def __len__(self):
        return len(self.samples)

    @property
    def check_ins_ids(self) -> np.ndarray:
        """Official check_ins_id of each sample's target, in sample order.

        This is what lets a rank vector from this repo be joined to one from a
        foreign implementation of the same task.

        Sample order is a permutation of test-CSV row order in general: the loop
        above walks users, and any user whose first eval check-in has no prior
        history contributes no sample at all. On the official NYC files it happens
        to come out as the identity -- 9,074 samples, 9,074 rows, same sequence --
        because those files arrive sorted by (UserId, ts_utc) and every test user
        already has training history. That is a property of the data, not of this
        code, so do not index into the CSV by position; join on the id.

        Derived from self.samples rather than re-walking eval_data so it stays a
        projection of the thing it labels: if the loop above changes, this follows
        instead of quietly disagreeing with it.
        """
        return np.array([s["target"]["check_ins_id"] for s in self.samples], dtype=np.int64)

    def __getitem__(self, idx):
        s = self.samples[idx]
        packed = _pack(s["history"], self.max_history_len, self.default_lat,
                       self.default_lon, self.t_ref)
        return _tensors(*packed, s["target"], self.t_ref,
                        _pack_repeat_history(s["history"], self.repeat_history_len),
                        self.repeat_history_len)


def collate_fn(batch):
    keys = ["poi_ids", "ts_hours", "hour", "dow", "locations", "target_poi",
            "target_hour", "target_dow", "target_location"]
    if "repeat_hist" in batch[0]:
        keys = keys + ["repeat_hist"]
    out = {k: torch.stack([b[k] for b in batch]) for k in keys}
    out["seq_lengths"] = torch.stack([b["seq_len"] for b in batch])
    for k in ("neg_ids", "is_explore"):
        if k in batch[0]:
            out[k] = torch.stack([b[k] for b in batch])
    return out


def _worker_init(worker_id: int) -> None:
    """Give each dataloader worker its own numpy stream.

    torch seeds each worker's `torch` RNG but NOT numpy's. Our negative sampler
    is pure numpy, so without this every worker would draw the SAME negatives --
    a silent correctness bug that only appears once num_workers > 0.
    """
    seed = (torch.initial_seed() + worker_id) % (2 ** 32)
    np.random.seed(seed)


def create_dataloaders(data: Dict, config: Dict, num_workers: int = 0):
    t_ref = data.get("t_ref", 0.0)
    train_ds = POITrainDataset(data["train_data"], data["num_pois"], data["poi_popularity"],
                               config, data["default_lat"], data["default_lon"], t_ref)
    val_ds = POIEvalDataset(data["train_data"], data["val_data"], config,
                            data["default_lat"], data["default_lon"], t_ref)
    train_plus_val = {uid: traj + data["val_data"].get(uid, [])
                      for uid, traj in data["train_data"].items()}
    test_ds = POIEvalDataset(train_plus_val, data["test_data"], config,
                             data["default_lat"], data["default_lon"], t_ref)

    ebs = config.get("eval_batch_size", config["batch_size"])
    mk = lambda ds, bs, sh: DataLoader(ds, batch_size=bs, shuffle=sh, collate_fn=collate_fn,
                                       num_workers=num_workers,
                                       worker_init_fn=_worker_init if num_workers else None,
                                       persistent_workers=num_workers > 0,
                                       pin_memory=torch.cuda.is_available())
    print(f"[data] samples: train={len(train_ds):,} val={len(val_ds):,} test={len(test_ds):,}")
    return (mk(train_ds, config["batch_size"], True), mk(val_ds, ebs, False), mk(test_ds, ebs, False))


Writing castpoi/data.py


In [10]:
%%writefile castpoi/layers.py
"""Shared encoders and losses used by the CaST-POI ranker."""
import math
from typing import Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F


def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1_r, lon1_r = torch.deg2rad(lat1), torch.deg2rad(lon1)
    lat2_r, lon2_r = torch.deg2rad(lat2), torch.deg2rad(lon2)
    dlat, dlon = lat2_r - lat1_r, lon2_r - lon1_r
    a = torch.sin(dlat / 2) ** 2 + torch.cos(lat1_r) * torch.cos(lat2_r) * torch.sin(dlon / 2) ** 2
    return R * 2 * torch.asin(torch.sqrt(torch.clamp(a, 0, 1)))


class TemporalEncoding(nn.Module):
    """Periodic time features from local hour-of-day and day-of-week plus a slot embedding."""

    def __init__(self, slot_embed_dim: int = 16, num_slots: int = 4, dropout: float = 0.1):
        super().__init__()
        self.omega_h = 2 * math.pi / 24.0
        self.omega_w = 2 * math.pi / 7.0
        self.slot_embedding = nn.Embedding(num_slots, slot_embed_dim)
        self.dropout = nn.Dropout(dropout)
        self.output_dim = 4 + slot_embed_dim
        nn.init.normal_(self.slot_embedding.weight, std=0.02)

    @staticmethod
    def time_slot(hour: torch.Tensor) -> torch.Tensor:
        slot = torch.full(hour.shape, 3, dtype=torch.long, device=hour.device)
        slot[(hour >= 6) & (hour < 12)] = 0
        slot[(hour >= 12) & (hour < 18)] = 1
        slot[hour >= 18] = 2
        return slot

    def forward(self, hour: torch.Tensor, dow: torch.Tensor) -> torch.Tensor:
        h, w = hour.float(), dow.float()
        feats = torch.stack([
            torch.sin(self.omega_h * h), torch.cos(self.omega_h * h),
            torch.sin(self.omega_w * w), torch.cos(self.omega_w * w),
        ], dim=-1)
        return self.dropout(torch.cat([feats, self.slot_embedding(self.time_slot(h))], dim=-1))


class SpatialEncoding(nn.Module):
    """Displacement features from the previous POI: log distance, a bucket embedding, and dlat/dlon."""

    DIST_BUCKETS = [0.0, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 50.0, float("inf")]

    def __init__(self, output_dim: int = 32, num_dist_buckets: int = 8,
                 dist_embed_dim: int = 16, dropout: float = 0.1):
        super().__init__()
        self.num_dist_buckets = num_dist_buckets
        self.dist_embedding = nn.Embedding(num_dist_buckets, dist_embed_dim)
        self.register_buffer("_edges", torch.tensor(self.DIST_BUCKETS[1:-1]), persistent=False)
        self.projection = nn.Sequential(
            nn.Linear(1 + dist_embed_dim + 2, output_dim), nn.ReLU(),
            nn.Linear(output_dim, output_dim), nn.Dropout(dropout),
        )
        self.output_dim = output_dim
        nn.init.normal_(self.dist_embedding.weight, std=0.02)

    def _bucket(self, d: torch.Tensor) -> torch.Tensor:
        return torch.bucketize(d, self._edges).clamp_(max=self.num_dist_buckets - 1)

    def forward(self, locations: torch.Tensor, prev_locations: torch.Tensor) -> torch.Tensor:
        lat1, lon1 = prev_locations[..., 0], prev_locations[..., 1]
        lat2, lon2 = locations[..., 0], locations[..., 1]
        dist = haversine_distance(lat1, lon1, lat2, lon2)
        feats = torch.cat([
            torch.log1p(dist).unsqueeze(-1),
            self.dist_embedding(self._bucket(dist)),
            ((lat2 - lat1) / 0.1).unsqueeze(-1),
            ((lon2 - lon1) / 0.1).unsqueeze(-1),
        ], dim=-1)
        return self.projection(feats)


class InputEncoder(nn.Module):
    """POI embedding concatenated with temporal and spatial encodings."""

    def __init__(self, num_pois, poi_embed_dim, slot_embed_dim, spatial_dim,
                 num_dist_buckets, dist_embed_dim, dropout):
        super().__init__()
        self.poi_embedding = nn.Embedding(num_pois, poi_embed_dim, padding_idx=0)
        nn.init.xavier_uniform_(self.poi_embedding.weight)
        with torch.no_grad():
            self.poi_embedding.weight[0].zero_()
        self.temporal_enc = TemporalEncoding(slot_embed_dim, dropout=dropout)
        self.spatial_enc = SpatialEncoding(spatial_dim, num_dist_buckets, dist_embed_dim, dropout)
        self.dropout = nn.Dropout(dropout)
        self.output_dim = poi_embed_dim + self.temporal_enc.output_dim + spatial_dim

    def forward(self, poi_ids, hour, dow, locations, prev_locations):
        return self.dropout(torch.cat([
            self.poi_embedding(poi_ids),
            self.temporal_enc(hour, dow),
            self.spatial_enc(locations, prev_locations),
        ], dim=-1))


def repeat_features(poi_ids: torch.Tensor, num_pois: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Per-candidate revisit signals from the visible history: log visit count, recency, visited flag.

    History is left-padded with 0; index 0 is the padding slot and is zeroed out.
    """
    B, L = poi_ids.shape
    valid = (poi_ids != 0).to(torch.float32)

    counts = torch.zeros(B, num_pois, device=poi_ids.device, dtype=torch.float32)
    counts.scatter_add_(1, poi_ids, valid)
    counts[:, 0] = 0.0

    pos = torch.arange(1, L + 1, device=poi_ids.device, dtype=torch.float32) / L
    pos = pos.unsqueeze(0).expand(B, L) * valid
    recency = torch.zeros(B, num_pois, device=poi_ids.device, dtype=torch.float32)
    recency.scatter_reduce_(1, poi_ids, pos, reduce="amax", include_self=True)
    recency[:, 0] = 0.0

    return torch.log1p(counts), recency, (counts > 0).to(torch.float32)


class RepeatGate(nn.Module):
    """Map the sequence state to three (unconstrained) weights over the revisit features."""

    def __init__(self, in_dim: int, hidden: int = 32):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, hidden), nn.ReLU(), nn.Linear(hidden, 3))
        nn.init.zeros_(self.net[-1].bias)
        nn.init.normal_(self.net[-1].weight, std=0.01)

    def forward(self, x):
        return self.net(x)


class CELoss(nn.Module):
    """Cross-entropy with label smoothing and a hard-negative BPR margin.

    forward() scores a sampled candidate set; forward_full() scores the full vocabulary.
    """

    def __init__(self, label_smoothing=0.02, explore_weight=1.5, bpr_weight=0.5, margin=1.0):
        super().__init__()
        self.label_smoothing = label_smoothing
        self.explore_weight = explore_weight
        self.bpr_weight = bpr_weight
        self.margin = margin

    def forward(self, pos_scores, neg_scores, is_explore: Optional[torch.Tensor] = None):
        logits = torch.cat([pos_scores.unsqueeze(1), neg_scores], dim=1)
        B, C = logits.shape
        log_probs = F.log_softmax(logits, dim=1)
        if self.label_smoothing > 0:
            smooth = self.label_smoothing / C
            target = torch.full_like(log_probs, smooth)
            target[:, 0] = 1.0 - self.label_smoothing + smooth
        else:
            target = torch.zeros_like(log_probs)
            target[:, 0] = 1.0
        ce = -(target * log_probs).sum(1)

        k = min(10, neg_scores.size(1))
        hard_neg, _ = neg_scores.topk(k, dim=1)
        bpr = F.relu(self.margin - (pos_scores.unsqueeze(1) - hard_neg)).mean(1)

        loss = ce + self.bpr_weight * bpr
        if is_explore is not None and self.explore_weight > 1.0:
            loss = loss * (1.0 + (self.explore_weight - 1.0) * is_explore)
        return loss.mean()

    def forward_full(self, logits, target_ids, is_explore: Optional[torch.Tensor] = None):
        B, V = logits.shape
        log_probs = F.log_softmax(logits, dim=1)
        tgt_col = target_ids.unsqueeze(1)

        target = torch.zeros_like(log_probs)
        if self.label_smoothing > 0:
            smooth = self.label_smoothing / (V - 1)
            target[:, 1:] = smooth
            target.scatter_(1, tgt_col, 1.0 - self.label_smoothing + smooth)
        else:
            target.scatter_(1, tgt_col, 1.0)
        ce = -(target * log_probs).sum(1)

        masked = logits.scatter(1, tgt_col, float("-inf"))
        masked[:, 0] = float("-inf")
        k = min(10, V - 2)
        hard_neg, _ = masked.topk(k, dim=1)
        pos_scores = logits.gather(1, tgt_col).squeeze(1)
        bpr = F.relu(self.margin - (pos_scores.unsqueeze(1) - hard_neg)).mean(1)

        loss = ce + self.bpr_weight * bpr
        if is_explore is not None and self.explore_weight > 1.0:
            loss = loss * (1.0 + (self.explore_weight - 1.0) * is_explore)
        return loss.mean()




Writing castpoi/model.py


In [12]:
%%writefile castpoi/engine.py
"""Training and evaluation loops.

Per-epoch history, per-sample test ranks and timing are written to disk so that
every reported number can be recomputed from a file rather than from memory.
"""
import math
import time
from pathlib import Path
from typing import Dict, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn

from .metrics import metrics_from_ranks, per_sample_ranks, summarize, format_metrics


class WarmupCosineScheduler:
    def __init__(self, optimizer, warmup_epochs: int, total_epochs: int, eta_min: float = 1e-6):
        self.optimizer = optimizer
        self.warmup_epochs = warmup_epochs
        self.total_epochs = total_epochs
        self.eta_min = eta_min
        self.base_lrs = [pg["lr"] for pg in optimizer.param_groups]
        self.epoch = 0

    def step(self):
        self.epoch += 1
        if self.epoch <= self.warmup_epochs:
            factor = self.epoch / max(self.warmup_epochs, 1)
        else:
            progress = (self.epoch - self.warmup_epochs) / max(self.total_epochs - self.warmup_epochs, 1)
            factor = 0.5 * (1 + math.cos(math.pi * min(progress, 1.0)))
        for pg, base in zip(self.optimizer.param_groups, self.base_lrs):
            pg["lr"] = self.eta_min + (base - self.eta_min) * factor

    def get_lr(self) -> float:
        return self.optimizer.param_groups[0]["lr"]


def _query_spatial_context(batch: Dict[str, torch.Tensor]):
    """Current position and previous position, both taken from the history.

    An earlier version passed `target_location`, the coordinates of the POI being
    predicted. SpatialEncoding turns that into the displacement to the answer,
    which is future information; results produced before protocol_rev 2 carry
    that leak. Sequences are left-padded, so column -1 is the most recent real
    check-in and -2 the one before it.

    A user with a single check-in of history has no column -2, only padding filled
    with the dataset mean coordinate. Those users get prev := current, i.e. zero
    displacement, rather than a fictitious move from the centroid.
    """
    cur = batch["locations"][:, -1, :]
    prev = batch["locations"][:, -2, :]
    has_prev = (batch["seq_lengths"] >= 2).unsqueeze(-1).to(cur.dtype)
    return cur, prev * has_prev + cur * (1 - has_prev)


def _to_device(batch, device):
    return {k: v.to(device, non_blocking=True) for k, v in batch.items()}


def _forward(model, batch):
    return model(
        poi_ids=batch["poi_ids"],
        ts_hours=batch["ts_hours"],
        hour=batch["hour"],
        dow=batch["dow"],
        locations=batch["locations"],
        seq_lengths=batch["seq_lengths"],
        query_hour=batch["target_hour"],
        query_dow=batch["target_dow"],
        query_location=_query_spatial_context(batch)[0],
        prev_query_location=_query_spatial_context(batch)[1],
        repeat_hist=batch.get("repeat_hist"),
    )


def train_epoch(model, loader, optimizer, criterion, device, config) -> float:
    model.train()
    total, n = 0.0, 0
    for batch in loader:
        batch = _to_device(batch, device)
        h_proj, _, rep = _forward(model, batch)
        if config.get("train_objective", "sampled") == "full":
            # Same scoring path evaluation uses, so training and testing now pose
            # the model the identical |L|-way problem.
            logits = model.compute_all_scores(h_proj, rep)
            loss = criterion.forward_full(logits, batch["target_poi"], batch.get("is_explore"))
        else:
            pos, neg = model.compute_sampled_scores(h_proj, batch["target_poi"], batch["neg_ids"], rep)
            loss = criterion(pos, neg, batch.get("is_explore"))
        if not torch.isfinite(loss):
            raise RuntimeError(
                "training loss became NaN/Inf. Refusing to continue: a diverged model "
                "scores NaN, and NaN ranks as position 1 for every sample, which would "
                "be reported as a flawless 100% HR@k. Lower the learning rate, or check "
                "the device (Apple MPS has produced NaN here where CPU and CUDA do not).")

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        if config["gradient_clip"] > 0:
            nn.utils.clip_grad_norm_(model.parameters(), config["gradient_clip"])
        optimizer.step()
        total += loss.item()
        n += 1
    return total / max(n, 1)


TOPK_KEEP = 20


@torch.no_grad()
def evaluate(model, loader, device, ks=(5, 10, 20), collect_alpha: bool = False,
             collect_topk: bool = False):
    """Returns (summary, per_sample_dict, extras). per_sample arrays enable paired tests."""
    model.eval()
    ranks, alphas, targets, topk = [], [], [], []
    for batch in loader:
        batch = _to_device(batch, device)
        h_proj, alpha, rep = _forward(model, batch)
        scores = model.compute_all_scores(h_proj, rep)
        ranks.append(per_sample_ranks(scores, batch["target_poi"]).cpu().numpy())
        targets.append(batch["target_poi"].cpu().numpy())
        # Rank-based metrics are recoverable from `ranks` alone, but coverage,
        # novelty and popularity bias need the predicted ids. Off by default
        # because validation runs this every epoch and discards the result;
        # run.py enables it for the single test evaluation that is saved.
        if collect_topk:
            topk.append(scores.topk(min(TOPK_KEEP, scores.size(1)),
                                    dim=1).indices.cpu().numpy().astype(np.int32))
        if collect_alpha:
            alphas.append(alpha.cpu().numpy())

    ranks = np.concatenate(ranks).astype(np.float64)
    per_sample = metrics_from_ranks(ranks, ks)
    extras = {"ranks": ranks, "targets": np.concatenate(targets),
              "topk": np.concatenate(topk) if topk else None}

    # Official check_ins_id per sample, so these ranks can be joined to a foreign
    # implementation's. Position i of `ranks` means sample i of the dataset only
    # because the eval loaders are built with shuffle=False; under a shuffling
    # sampler the join would still run and would pair every sample with the wrong
    # id, so check rather than assume.
    ds = getattr(loader, "dataset", None)
    if hasattr(ds, "check_ins_ids"):
        if not isinstance(loader.sampler, torch.utils.data.SequentialSampler):
            raise RuntimeError(
                f"eval loader uses {type(loader.sampler).__name__}, not SequentialSampler. "
                f"Rank i would not correspond to sample i, so check_ins_id would mislabel "
                f"every row and any paired test built on it would be silently wrong.")
        cids = ds.check_ins_ids
        if len(cids) != len(ranks):
            raise RuntimeError(f"{len(cids)} check_ins_ids vs {len(ranks)} ranks.")
        extras["check_ins_id"] = cids
    if collect_alpha:
        extras["alpha"] = np.concatenate(alphas, axis=0)
    return summarize(per_sample), per_sample, extras


def train_model(model, train_loader, val_loader, config, device, logger=None) -> Tuple[nn.Module, Dict]:
    """Train with warmup+cosine LR and early stopping on validation HR@10."""
    from .layers import CELoss

    log = (logger.info if logger else print)
    model = model.to(device)
    criterion = CELoss(config["label_smoothing"], config["explore_weight"],
                           config.get("bpr_weight", 0.5), config.get("bpr_margin", 1.0))
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["learning_rate"],
                                  weight_decay=config["weight_decay"])
    scheduler = WarmupCosineScheduler(optimizer, config["warmup_epochs"], config["num_epochs"])

    history = {"epochs": [], "best_epoch": None, "stopped_early": False}
    best_hr10, best_state, patience = -1.0, None, 0
    t_start = time.time()

    for epoch in range(1, config["num_epochs"] + 1):
        t0 = time.time()
        loss = train_epoch(model, train_loader, optimizer, criterion, device, config)
        train_s = time.time() - t0
        scheduler.step()

        t1 = time.time()
        val_metrics, _, _ = evaluate(model, val_loader, device, config["eval_ks"])
        eval_s = time.time() - t1

        history["epochs"].append({
            "epoch": epoch, "train_loss": loss, "lr": scheduler.get_lr(),
            "train_seconds": train_s, "eval_seconds": eval_s,
            "val": val_metrics,
        })
        log(f"epoch {epoch:3d} | loss {loss:.4f} | lr {scheduler.get_lr():.2e} | "
            f"{train_s:.1f}s+{eval_s:.1f}s | val {format_metrics(val_metrics)}")

        improved = val_metrics["HR@10"] > best_hr10
        if improved:
            best_hr10 = val_metrics["HR@10"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            history["best_epoch"] = epoch
            patience = 0
        elif epoch > config["warmup_epochs"]:
            patience += 1
            if patience >= config["early_stopping_patience"]:
                log(f"early stop at epoch {epoch} (best val HR@10 {best_hr10 * 100:.2f} @ epoch {history['best_epoch']})")
                history["stopped_early"] = True
                break

    history["total_train_seconds"] = time.time() - t_start
    history["best_val_hr10"] = best_hr10
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history


@torch.no_grad()
def measure_inference(model, loader, device, batch_size: int, n_warmup: int = 3, n_iters: int = 20) -> Dict:
    """Honest latency/throughput: measured separately, not derived one from the other."""
    model.eval()
    batch = _to_device(next(iter(loader)), device)
    B = batch["poi_ids"].size(0)

    for _ in range(n_warmup):
        h, _, rep = _forward(model, batch)
        model.compute_all_scores(h, rep)
    if device.type == "cuda":
        torch.cuda.synchronize()

    times = []
    for _ in range(n_iters):
        t0 = time.perf_counter()
        h, _, rep = _forward(model, batch)
        model.compute_all_scores(h, rep)
        if device.type == "cuda":
            torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)

    times = np.array(times)
    per_batch_ms = float(times.mean() * 1000)
    out = {
        "batch_size": B,
        "batch_latency_ms_mean": per_batch_ms,
        "batch_latency_ms_std": float(times.std(ddof=1) * 1000) if len(times) > 1 else 0.0,
        "per_query_latency_ms": per_batch_ms / B,
        "throughput_queries_per_s": B / (per_batch_ms / 1000),
        "n_iters": n_iters,
        "device": str(device),
    }
    if device.type == "cuda":
        out["peak_memory_gb"] = torch.cuda.max_memory_allocated() / 1024 ** 3
    return out




Writing castpoi/engine.py


### 3. Write CaST-POI v2 + build_loo_split + runner

In [13]:
%%writefile castpoi/model.py
"""CaST-POI: candidate-conditioned spatiotemporal ranker for next-POI recommendation."""
import math
import torch
import torch.nn as nn

from .layers import InputEncoder, RepeatGate, repeat_features

TIME_BOUNDS_H = [0.0, 1.0, 6.0, 24.0, 168.0, 720.0, 1e9]
DIST_BOUNDS_KM = [0.0, 0.5, 2.0, 5.0, 20.0, 100.0, 1e9]


def _bucketize(x, bounds):
    out = torch.zeros_like(x, dtype=torch.long)
    for i in range(len(bounds) - 1):
        out = torch.where((x >= bounds[i]) & (x < bounds[i + 1]), torch.full_like(out, i), out)
    return out


def _haversine_km(a, b):
    R = 6371.0
    lat1, lon1 = torch.deg2rad(a[..., 0]), torch.deg2rad(a[..., 1])
    lat2, lon2 = torch.deg2rad(b[..., 0]), torch.deg2rad(b[..., 1])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    h = torch.sin(dlat / 2) ** 2 + torch.cos(lat1) * torch.cos(lat2) * torch.sin(dlon / 2) ** 2
    return R * 2 * torch.asin(torch.sqrt(torch.clamp(h, 0, 1)))


class _PointWiseFFN(nn.Module):
    def __init__(self, d, dropout):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, d), nn.ReLU(), nn.Dropout(dropout), nn.Linear(d, d))

    def forward(self, x):
        return x + self.net(x)


class CaSTPOI(nn.Module):
    def __init__(self, num_pois, poi_locations, config):
        super().__init__()

        d = config["poi_embed_dim"]
        self.d = d
        self.num_pois = num_pois
        self.K = 1
        self.H = config.get("castpoi_heads", 4)
        self.dh = d // self.H
        self.n_backbone = config.get("castpoi_backbone_layers", 2)
        self.n_cross = config.get("castpoi_cross_layers", 2)
        self.cand_chunk = config.get("castpoi_cand_chunk", 1024)
        max_len = config["max_history_len"]

        self.use_backbone = config.get("use_backbone", True)
        self.candidate_conditioned = config.get("castpoi_candidate_conditioned", True)
        self.use_temporal_bias = config.get("castpoi_temporal_bias", True)
        self.use_spatial_bias = config.get("castpoi_spatial_bias", True)
        self.use_repeat = config.get("use_repeat", True)

        self.input_encoder = InputEncoder(
            num_pois, d, config["slot_embed_dim"], config["spatial_dim"],
            config["num_dist_buckets"], config["dist_embed_dim"], config["dropout"])
        self.poi_embedding = self.input_encoder.poi_embedding
        self.in_proj = nn.Linear(self.input_encoder.output_dim, d)
        self.pos_embedding = nn.Embedding(max_len, d)
        nn.init.normal_(self.pos_embedding.weight, std=0.02)
        self.emb_drop = nn.Dropout(config["dropout"])

        # causal self-attention backbone
        self.bb_ln1 = nn.ModuleList(nn.LayerNorm(d) for _ in range(self.n_backbone))
        self.bb_q = nn.ModuleList(nn.Linear(d, d) for _ in range(self.n_backbone))
        self.bb_k = nn.ModuleList(nn.Linear(d, d) for _ in range(self.n_backbone))
        self.bb_v = nn.ModuleList(nn.Linear(d, d) for _ in range(self.n_backbone))
        self.bb_o = nn.ModuleList(nn.Linear(d, d) for _ in range(self.n_backbone))
        self.bb_ln2 = nn.ModuleList(nn.LayerNorm(d) for _ in range(self.n_backbone))
        self.bb_ffn = nn.ModuleList(_PointWiseFFN(d, config["dropout"]) for _ in range(self.n_backbone))
        self.last_ln = nn.LayerNorm(d)

        # candidate-conditioned cross-attention
        self.cq = nn.ModuleList(nn.Linear(d, d) for _ in range(self.n_cross))
        self.ck = nn.ModuleList(nn.Linear(d, d) for _ in range(self.n_cross))
        self.cv = nn.ModuleList(nn.Linear(d, d) for _ in range(self.n_cross))
        self.co = nn.ModuleList(nn.Linear(d, d) for _ in range(self.n_cross))
        self.cln1 = nn.ModuleList(nn.LayerNorm(d) for _ in range(self.n_cross))
        self.cln2 = nn.ModuleList(nn.LayerNorm(d) for _ in range(self.n_cross))
        self.cffn = nn.ModuleList(_PointWiseFFN(d, config["dropout"]) for _ in range(self.n_cross))
        self.t_bias = nn.Embedding(len(TIME_BOUNDS_H) - 1, 1)
        self.s_bias = nn.Embedding(len(DIST_BOUNDS_KM) - 1, 1)
        nn.init.zeros_(self.t_bias.weight)
        nn.init.zeros_(self.s_bias.weight)
        self.cand_head = nn.Sequential(
            nn.Linear(3 * d, d), nn.GELU(), nn.Dropout(config["dropout"]), nn.Linear(d, 1))
        self.cand_gate = nn.Parameter(torch.tensor(0.0))

        # repeat gate
        if self.use_repeat:
            self.repeat_gate = RepeatGate(d, config.get("repeat_gate_hidden", 32))
            self.register_buffer("_pad_onehot", torch.zeros(num_pois), persistent=False)
            self._pad_onehot[0] = 1.0

        self.poi_bias = nn.Parameter(torch.zeros(num_pois))
        self.register_buffer("poi_locations", torch.as_tensor(poi_locations, dtype=torch.float32))
        pad = torch.zeros(num_pois); pad[0] = -1e9
        self.register_buffer("_pad_mask", pad, persistent=False)

    def _split(self, t):
        B, N, _ = t.shape
        return t.view(B, N, self.H, self.dh)

    def forward(self, poi_ids, ts_hours, hour, dow, locations, seq_lengths,
                query_hour=None, query_dow=None, query_location=None,
                prev_query_location=None, repeat_hist=None):
        B, L = poi_ids.shape
        pad = (poi_ids == 0)
        prev = torch.empty_like(locations)
        prev[:, 1:] = locations[:, :-1]; prev[:, 0] = locations[:, 0]
        x = self.in_proj(self.input_encoder(poi_ids, hour, dow, locations, prev))
        pos = torch.arange(L, device=poi_ids.device).unsqueeze(0).expand(B, L)
        x = self.emb_drop(x + self.pos_embedding(pos)).masked_fill(pad.unsqueeze(-1), 0.0)

        causal = torch.triu(torch.ones(L, L, dtype=torch.bool, device=poi_ids.device), 1)
        block = causal.view(1, 1, L, L) | pad.view(B, 1, 1, L)
        h = x
        if self.use_backbone:
            for l in range(self.n_backbone):
                q = self._split(self.bb_q[l](self.bb_ln1[l](h))).transpose(1, 2)
                k = self._split(self.bb_k[l](h)).transpose(1, 2)
                v = self._split(self.bb_v[l](h)).transpose(1, 2)
                logit = (q @ k.transpose(-1, -2)) / math.sqrt(self.dh)
                logit = logit.masked_fill(block, -1e9)
                a = torch.softmax(logit, dim=-1) @ v
                h = h + self.bb_o[l](a.transpose(1, 2).reshape(B, L, self.d))
                h = self.bb_ffn[l](self.bb_ln2[l](h)).masked_fill(pad.unsqueeze(-1), 0.0)
            h = self.last_ln(h)
        h_seq = h[:, -1, :]

        Ks = [self._split(self.ck[l](h)).transpose(1, 2) for l in range(self.n_cross)]
        Vs = [self._split(self.cv[l](h)).transpose(1, 2) for l in range(self.n_cross)]

        time_bias = None
        if self.use_temporal_bias:
            ref = ts_hours.gather(1, (L - 1) * torch.ones(B, 1, dtype=torch.long, device=poi_ids.device))
            dt = (ref - ts_hours).clamp(min=0)
            time_bias = self.t_bias(_bucketize(dt, TIME_BOUNDS_H)).squeeze(-1)

        rep = None
        if self.use_repeat:
            src = repeat_hist if repeat_hist is not None else poi_ids
            cnt, rec, vis = repeat_features(src, self.num_pois)
            w = self.repeat_gate(h_seq)
            rep = w[:, 0:1] * cnt + w[:, 1:2] * rec + w[:, 2:3] * vis
            rep = rep * (1.0 - self._pad_onehot)

        cache = {"h_seq": h_seq, "Ks": Ks, "Vs": Vs, "time_bias": time_bias,
                 "pad": pad, "hist_locs": locations, "B": B, "L": L}
        return cache, None, rep

    def _cand_cond_score(self, cache, cand_ids, cand_locs):
        B, C = cand_ids.shape
        L = cache["L"]
        e_cand = self.poi_embedding(cand_ids)
        kpm = cache["pad"].view(B, 1, 1, L)
        bias = None
        if cache["time_bias"] is not None:
            bias = cache["time_bias"].view(B, 1, 1, L)
        if self.use_spatial_bias:
            hh = cache["hist_locs"].unsqueeze(1); cc = cand_locs.unsqueeze(2)
            sb = self.s_bias(_bucketize(_haversine_km(hh, cc), DIST_BOUNDS_KM)).squeeze(-1).unsqueeze(1)
            bias = sb if bias is None else bias + sb
        ur = e_cand
        for l in range(self.n_cross):
            q = self._split(self.cq[l](ur)).transpose(1, 2)
            logit = (q @ cache["Ks"][l].transpose(-1, -2)) / math.sqrt(self.dh)
            if bias is not None:
                logit = logit + bias
            logit = logit.masked_fill(kpm, -1e9)
            attn = torch.softmax(logit, dim=-1)
            out = (attn @ cache["Vs"][l]).transpose(1, 2).reshape(B, C, self.d)
            ur = self.cln1[l](ur + self.co[l](out))
            ur = self.cln2[l](self.cffn[l](ur))
        feat = torch.cat([ur, e_cand, ur * e_cand], dim=-1)
        return self.cand_head(feat).squeeze(-1)

    def _score(self, cache, cand_ids, cand_locs):
        e_cand = self.poi_embedding(cand_ids)
        s = (cache["h_seq"].unsqueeze(1) * e_cand).sum(-1)
        s = s + self.poi_bias[cand_ids]
        if self.candidate_conditioned:
            s = s + self.cand_gate * self._cand_cond_score(cache, cand_ids, cand_locs)
        return s

    def compute_sampled_scores(self, cache, pos_ids, neg_ids, rep=None):
        cand = torch.cat([pos_ids.unsqueeze(1), neg_ids], dim=1)
        s = self._score(cache, cand, self.poi_locations[cand])
        if rep is not None:
            s = s + rep.gather(1, cand)
        return s[:, 0], s[:, 1:]

    def compute_all_scores(self, cache, rep=None):
        B = cache["B"]
        device = cache["h_seq"].device
        scores = torch.full((B, self.num_pois), -1e9, device=device)
        for lo in range(1, self.num_pois, self.cand_chunk):
            hi = min(lo + self.cand_chunk, self.num_pois)
            ids = torch.arange(lo, hi, device=device)
            cand = ids.unsqueeze(0).expand(B, hi - lo)
            cand_locs = self.poi_locations[lo:hi].unsqueeze(0).expand(B, hi - lo, 2)
            scores[:, lo:hi] = self._score(cache, cand, cand_locs)
        if rep is not None:
            scores = scores + rep + self._pad_mask
        return scores


Writing castpoi_ranker.py


In [14]:
%%writefile build_loo_split.py
#!/usr/bin/env python3
"""Build a per-user leave-one-out (LOO) split from the chronological data_official,
in the EXACT format castpoi/official.py reads, so CaST-POI and (via export) RecBole
run on one bit-identical LOO split.

Per user, chronological by UTCTimeOffset:
    last check-in     -> test
    second-to-last    -> validation
    all earlier       -> train
Users with < 3 check-ins are dropped (LOO needs >= 1 train + 1 val + 1 test).
val/test rows whose PoiId is not in the NEW train are removed (unseen-POI removal,
matching the upstream pipeline and the range assertion in official.py). Every
original column is preserved, so official.py loads it unchanged.

    python build_loo_split.py --src data_official --out data_loo --datasets nyc tky ca
"""
import argparse
import os
import pandas as pd

FILES = {"train": "train_sample.csv",
         "val":   "validate_sample_with_traj.csv",
         "test":  "test_sample_with_traj.csv"}


def build(ds, src, out):
    # 1. read + concat the three chronological splits (identical columns, verified)
    parts = [pd.read_csv(os.path.join(src, ds, fn), low_memory=False)
             for fn in FILES.values()]
    df = pd.concat(parts, ignore_index=True)
    n0 = len(df)

    # 2. chronological order per user. UTCTimeOffset is a wall-clock string; parse
    #    the first 19 chars exactly as timeparse.py does. Stable sort keeps file
    #    order on exact ties (matches official.py's mergesort on ts_utc).
    df["_t"] = pd.to_datetime(df["UTCTimeOffset"].astype(str).str.slice(0, 19),
                              format="%Y-%m-%d %H:%M:%S", errors="coerce")
    assert df["_t"].notna().all(), f"{ds}: some UTCTimeOffset failed to parse"
    df = df.sort_values(["UserId", "_t"], kind="mergesort").reset_index(drop=True)

    # 3. LOO tag: rank from the end within each user (0 = last check-in)
    df["_rk"] = df.groupby("UserId").cumcount(ascending=False)
    df["_n"] = df.groupby("UserId")["UserId"].transform("size")
    df = df[df["_n"] >= 3].copy()
    df["_split"] = "train"
    df.loc[df["_rk"] == 0, "_split"] = "test"
    df.loc[df["_rk"] == 1, "_split"] = "val"

    # 4. unseen removal: drop val/test rows whose PoiId or UserId is absent from
    #    the new train (keeps official.py's range assertion satisfied).
    train_pois = set(df.loc[df["_split"] == "train", "PoiId"].unique())
    train_users = set(df.loc[df["_split"] == "train", "UserId"].unique())
    is_train = df["_split"] == "train"
    seen = df["PoiId"].isin(train_pois) & df["UserId"].isin(train_users)
    df = df[is_train | seen].copy()

    # 5. write, original columns only, in official.py's expected filenames
    orig_cols = [c for c in df.columns if not c.startswith("_")]
    os.makedirs(os.path.join(out, ds), exist_ok=True)
    counts = {}
    for split, fn in FILES.items():
        sub = df.loc[df["_split"] == split, orig_cols]
        sub.to_csv(os.path.join(out, ds, fn), index=False)
        counts[split] = len(sub)

    n_users = df["UserId"].nunique()
    print(f"[{ds}] from {n0:,} check-ins -> train {counts['train']:,} / "
          f"val {counts['val']:,} / test {counts['test']:,}  "
          f"| users(with train) {n_users:,} | |L|~{len(train_pois):,}")
    print(f"[{ds}] LOO sanity: test ({counts['test']:,}) should be <= #users "
          f"({n_users:,}); one test target per user minus unseen-POI drops.")
    return counts, n_users


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--src", default="data_official")
    ap.add_argument("--out", default="data_loo")
    ap.add_argument("--datasets", nargs="+", default=["nyc", "tky", "ca"])
    a = ap.parse_args()
    for ds in a.datasets:
        build(ds, a.src, a.out)
    print("\nDone. Point the harness at it:  --official-dir", a.out)


if __name__ == "__main__":
    main()


Writing build_loo_split.py


In [15]:
%%writefile run_loo.py
#!/usr/bin/env python3
"""
CaST-POI v2 under the canonical per-user LOO split (kdd_baselines/build_loo_split.py)
+ full-vocabulary ranking. Byte-identical data to the RecBole baselines.

IMPORTANT: point OFFICIAL_DIR (or --official-dir) at the **data_loo** directory
produced by build_loo_split.py, NOT at data_official. This runner does NOT
re-split; it loads whatever split build_loo_split.py wrote (per-user LOO with
unseen-POI removal), so CaST-POI and the baselines run on one bit-identical split.

  python build_loo_split.py --src data_official --out data_loo --datasets nyc tky ca
  OFFICIAL_DIR=data_loo python run_loo.py --dataset all --seeds 42 43 44 --lr 1e-3
"""
import argparse, json, os, sys, time
from pathlib import Path
import numpy as np
import torch

HERE = Path(__file__).resolve().parent
RESULTS = Path(os.environ.get("CASTPOI_RESULTS", HERE / "results"))
# LOO data dir (built by build_loo_split.py). Defaults to ./data_loo; override with OFFICIAL_DIR.
OFFICIAL_DIR = Path(os.environ.get("OFFICIAL_DIR", HERE / "data_loo"))

from castpoi.config import resolve_config                       # noqa: E402
from castpoi.official import load_official                      # noqa: E402
from castpoi.data import create_dataloaders                     # noqa: E402
from castpoi.engine import train_model, evaluate, measure_inference  # noqa: E402
from castpoi.metrics import format_metrics                      # noqa: E402
from castpoi.utils import set_seed, pick_device, count_params   # noqa: E402
from castpoi.model import CaSTPOI                                # noqa: E402

DATASETS = ("nyc", "tky", "ca")
SEEDS = [42, 43, 44]


def build_config(dataset, quick, lr):
    cfg = resolve_config(dataset, {})
    cfg["eval_ks"] = [1, 5, 10, 20]
    cfg["eval_batch_size"] = 256
    cfg["castpoi_cand_chunk"] = 1024
    # full-CE matches every RecBole
    # baseline (train_neg_sample_args:None) use -> the only fair setting.
    cfg["train_objective"] = "full"
    cfg["batch_size"] = min(cfg["batch_size"], 128)
    if lr is not None:
        cfg["learning_rate"] = lr
    if quick:
        cfg["num_epochs"] = 2
    return cfg


# ---- ablation variants: config overrides toggling one component off each ----
# switches live in castpoi.model.py (v2_use_repeat / castpoi_candidate_conditioned /
# castpoi_spatial_bias / castpoi_temporal_bias / v2_use_backbone).
ABLATION_VARIANTS = {
    "full":               {},
    "no_repeat":          {"v2_use_repeat": False},
    "no_candcond":        {"castpoi_candidate_conditioned": False},
    "no_spatial_bias":    {"castpoi_spatial_bias": False},
    "no_temporal_bias":   {"castpoi_temporal_bias": False},
    "no_backbone":        {"v2_use_backbone": False},
}


def run_one(dataset, data, cfg, seed, device, tag, variant="full"):
    out_dir = RESULTS / "runs" / tag / dataset / variant / f"seed{seed}"
    if (out_dir / "metrics.json").exists():
        print(f"[skip] {dataset}/seed{seed} already done")
        return json.load(open(out_dir / "metrics.json"))["test"]
    set_seed(seed)
    model = CaSTPOI(data["num_pois"], data["poi_locations"], cfg)
    tl, vl, tel = create_dataloaders(data, cfg)
    t0 = time.time()
    model, hist = train_model(model, tl, vl, cfg, device)
    test_metrics, _, extras = evaluate(model, tel, device, cfg["eval_ks"], collect_topk=True)
    wall = time.time() - t0

    ranks = extras.get("ranks"); rep_expl = None
    if ranks is not None:
        is_rep = []
        for b in tel:
            src = b.get("repeat_hist", b["poi_ids"])
            is_rep.append((src == b["target_poi"].unsqueeze(1)).any(1).cpu().numpy())
        is_rep = np.concatenate(is_rep).astype(bool)
        if len(is_rep) == len(ranks):
            out_dir.mkdir(parents=True, exist_ok=True)
            np.save(out_dir / "ranks.npy", ranks); np.save(out_dir / "is_repeat.npy", is_rep)
            if extras.get("check_ins_id") is not None:
                np.save(out_dir / "check_ins_id.npy", extras["check_ins_id"])
            def sub(m):
                r = ranks[m]
                if not len(r): return {"n": 0}
                return {**{f"HR@{k}": float((r <= k).mean()) for k in cfg["eval_ks"]},
                        "MRR": float((1/r).mean()), "n": int(len(r))}
            rep_expl = {"repeat": sub(is_rep), "explore": sub(~is_rep), "repeat_frac": float(is_rep.mean())}
    try:
        eff = measure_inference(model, tel, device, cfg["eval_batch_size"])
    except Exception as e:
        eff = {"error": str(e)}
    out_dir.mkdir(parents=True, exist_ok=True)
    rec = {"dataset": dataset, "model": "castpoi.model", "variant": variant, "split": "loo", "seed": seed,
           "num_pois": data["num_pois"], "data_fingerprint": data["stats"].get("data_fingerprint"),
           "params": count_params(model), "test": test_metrics, "repeat_explore": rep_expl,
           "efficiency": eff, "best_val_hr10": hist.get("best_val_hr10"), "wall_seconds": wall,
           "config": {k: cfg[k] for k in ("learning_rate", "batch_size", "num_epochs", "train_objective", "eval_ks")}}
    json.dump(rec, open(out_dir / "metrics.json", "w"), indent=2)
    print(f"[{dataset}/loo/seed{seed}] {format_metrics(test_metrics)} | {wall/60:.1f}min | fp={rec['data_fingerprint']}")
    return test_metrics


def run_dataset(dataset, seeds, device, quick, lr):
    tag = "castpoi.model_loo" + ("" if lr is None else f"_lr{lr:g}")
    print(f"\n{'#'*72}\n# CaST-POI v2 [LOO] on {dataset.upper()} | device={device} | seeds={seeds} "
          f"lr={lr or 'default'} | canonical per-user LOO + full-ranking\n{'#'*72}")
    data = load_official(dataset, OFFICIAL_DIR)          # <-- data_loo (already LOO-split)
    if not str(OFFICIAL_DIR).rstrip("/").endswith("data_loo"):
        print(f"[warn] OFFICIAL_DIR={OFFICIAL_DIR} does not look like a data_loo dir; "
              f"make sure it was produced by build_loo_split.py")
    cfg = build_config(dataset, quick, lr)
    ms = [run_one(dataset, data, cfg, s, device, tag) for s in seeds]
    agg = {k: {"mean": float(np.mean([m[k] for m in ms])), "std": float(np.std([m[k] for m in ms]))} for k in ms[0]}
    print("  == mean±std: " + " ".join(f"{k} {v['mean']*100:.2f}±{v['std']*100:.2f}" for k, v in agg.items()))
    RESULTS.mkdir(parents=True, exist_ok=True)
    json.dump({"dataset": dataset, "split": "loo", "protocol": "canonical per-user LOO + full-ranking",
               "data_fingerprint": data["stats"].get("data_fingerprint"), "seeds": seeds,
               "results": {"full": agg}}, open(RESULTS / f"castpoi.model_loo_{dataset}.json", "w"), indent=2)
    print(f"Saved results/castpoi.model_loo_{dataset}.json")


def run_ablation(dataset, seeds, device, quick, lr, variants):
    tag = "castpoi.model_loo_ablation" + ("" if lr is None else f"_lr{lr:g}")
    print(f"\n{'#'*72}\n# CaST-POI v2 [LOO ABLATION] on {dataset.upper()} | device={device} | "
          f"seeds={seeds} | variants={variants}\n{'#'*72}")
    data = load_official(dataset, OFFICIAL_DIR)
    base = build_config(dataset, quick, lr)
    summary = {}
    for v in variants:
        cfg = dict(base); cfg.update(ABLATION_VARIANTS[v])
        ms = [run_one(dataset, data, cfg, s, device, tag, variant=v) for s in seeds]
        agg = {k: {"mean": float(np.mean([m[k] for m in ms])),
                   "std": float(np.std([m[k] for m in ms]))} for k in ms[0]}
        summary[v] = agg
        hr10 = agg.get("HR@10", {}).get("mean", float("nan"))
        print(f"  [{v}] HR@10={hr10*100:.2f}")
    # deltas vs full (HR@10) for the ablation table
    if "full" in summary:
        base_hr = summary["full"]["HR@10"]["mean"]
        print("  == HR@10 vs full ==")
        for v in variants:
            d = (summary[v]["HR@10"]["mean"] - base_hr) * 100
            print(f"     {v:18} {summary[v]['HR@10']['mean']*100:6.2f}  Δ{d:+.2f}")
    RESULTS.mkdir(parents=True, exist_ok=True)
    json.dump({"dataset": dataset, "split": "loo", "kind": "ablation", "seeds": seeds,
               "data_fingerprint": data["stats"].get("data_fingerprint"), "results": summary},
              open(RESULTS / f"castpoi.model_loo_ablation_{dataset}.json", "w"), indent=2)
    print(f"Saved results/castpoi.model_loo_ablation_{dataset}.json")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dataset", default="nyc")
    ap.add_argument("--seeds", type=int, nargs="+", default=SEEDS)
    ap.add_argument("--device", default="auto")
    ap.add_argument("--quick", action="store_true")
    ap.add_argument("--lr", type=float, default=1e-3)
    ap.add_argument("--ablation", action="store_true",
                    help="run the LOO ablation (variants toggling one component off each)")
    ap.add_argument("--variants", nargs="+", default=list(ABLATION_VARIANTS),
                    help=f"ablation variants to run (default all): {list(ABLATION_VARIANTS)}")
    args = ap.parse_args()
    for v in args.variants:
        assert v in ABLATION_VARIANTS, f"unknown variant {v}; choose from {list(ABLATION_VARIANTS)}"
    RESULTS.mkdir(parents=True, exist_ok=True)
    assert OFFICIAL_DIR.exists(), (f"OFFICIAL_DIR={OFFICIAL_DIR} not found. Run "
        f"`python build_loo_split.py --src data_official --out data_loo` first and set OFFICIAL_DIR=data_loo.")
    device = pick_device(args.device)
    seeds = [42] if args.quick else args.seeds
    for ds in (list(DATASETS) if args.dataset == "all" else [args.dataset]):
        if args.ablation:
            run_ablation(ds, seeds, device, args.quick, args.lr, args.variants)
        else:
            run_dataset(ds, seeds, device, args.quick, args.lr)


if __name__ == "__main__":
    main()


Writing run_loo.py


### 4. Build canonical LOO split (data_loo)

In [16]:
# 4) Build the canonical per-user LOO split (data_loo) — SAME as the baselines
!python build_loo_split.py --src data_official --out data_loo --datasets nyc tky ca
print('data_loo:', sorted(os.listdir('data_loo')))

[nyc] from 101,760 check-ins -> train 99,666 / val 1,043 / test 1,044  | users(with train) 1,047 | |L|~4,974
[nyc] LOO sanity: test (1,044) should be <= #users (1,047); one test target per user minus unseen-POI drops.
[tky] from 403,148 check-ins -> train 398,586 / val 2,280 / test 2,280  | users(with train) 2,280 | |L|~7,832
[tky] LOO sanity: test (2,280) should be <= #users (2,280); one test target per user minus unseen-POI drops.
[ca] from 221,717 check-ins -> train 213,805 / val 3,956 / test 3,956  | users(with train) 3,956 | |L|~9,689
[ca] LOO sanity: test (3,956) should be <= #users (3,956); one test target per user minus unseen-POI drops.

Done. Point the harness at it:  --official-dir data_loo
data_loo: ['ca', 'nyc', 'tky']


### 5. Run v2 under LOO

In [17]:
# 5) Run CaST-POI v2 under LOO (this seed) on data_loo
!python run_loo.py --dataset all --seeds {SEED} --lr 1e-3 --device cuda


########################################################################
# CaST-POI v2 [LOO] on NYC | device=cuda | seeds=[44] lr=0.001 | canonical per-user LOO + full-ranking
########################################################################

[official] NYC (Foursquare New York City, Foursquare)
[official] provenance: GETNext pre-split NYC_{train,val,test}.csv
[official] train:  99,666 check-ins  (11.1 MB)
[official] val  :   1,043 check-ins  (0.1 MB)
[official] test :   1,044 check-ins  (0.1 MB)
[official] note: 6 vocabulary slots never appear in any split
[official] users=1,047 POIs=4,980 check-ins=101,753 trajectories=13,938
[official] local time OK (trough 04:00, day/night 14.4) via America/New_York, column read as local
[official] fingerprint 73c596f2920e5266
[skip] nyc/seed44 already done
  == mean±std: HR@1 26.25±0.00 NDCG@1 26.25±0.00 HR@5 51.05±0.00 NDCG@5 39.48±0.00 HR@10 57.85±0.00 NDCG@10 41.73±0.00 HR@20 61.11±0.00 NDCG@20 42.56±0.00 MRR 37.14±0.00
Saved results/ca

## 6b) LOO ablation — decides the story (method-forward vs critique-forward)
Fast decision run: **full vs no\_candcond** on all datasets (1 seed) answers "does candidate-conditioning help under LOO?". Drop `--variants` to run all 6 variants for the final table.

In [18]:
# FULL ablation table (all variants: full / no_repeat / no_candcond /
# no_spatial_bias / no_temporal_bias / no_backbone). One seed per Colab.
!python run_loo.py --dataset all --seeds {SEED} --lr 1e-3 --device cuda --ablation

# (fast direction-only check, already done):
# !python run_loo.py --dataset all --seeds {SEED} --lr 1e-3 --device cuda --ablation --variants full no_candcond



########################################################################
# CaST-POI v2 [LOO ABLATION] on NYC | device=cuda | seeds=[44] | variants=['full', 'no_repeat', 'no_candcond', 'no_spatial_bias', 'no_temporal_bias', 'no_backbone']
########################################################################

[official] NYC (Foursquare New York City, Foursquare)
[official] provenance: GETNext pre-split NYC_{train,val,test}.csv
[official] train:  99,666 check-ins  (11.1 MB)
[official] val  :   1,043 check-ins  (0.1 MB)
[official] test :   1,044 check-ins  (0.1 MB)
[official] note: 6 vocabulary slots never appear in any split
[official] users=1,047 POIs=4,980 check-ins=101,753 trajectories=13,938
[official] local time OK (trough 04:00, day/night 14.4) via America/New_York, column read as local
[official] fingerprint 73c596f2920e5266
[skip] nyc/seed44 already done
  [full] HR@10=56.99
[skip] nyc/seed44 already done
  [no_repeat] HR@10=54.69
[skip] nyc/seed44 already done
  [no_candcond]

### 6. Results (LOO) + repeat/explore

In [19]:
# 6) Aggregate finished LOO seeds + repeat/explore
import json, glob, os, numpy as np
R=os.environ['CASTPOI_RESULTS']
for ds in ['nyc','tky','ca']:
    fs=sorted(glob.glob(f"{R}/runs/castpoi.model_loo*/{ds}/full/seed*/metrics.json"))
    if not fs: continue
    recs=[json.load(open(f)) for f in fs]; _u={}; recs=[_u.setdefault(r['seed'],r) for r in recs if r['seed'] not in _u]
    seeds=[r['seed'] for r in recs]; fp=recs[0].get('data_fingerprint')
    def ag(k): v=[r['test'][k]*100 for r in recs]; return np.mean(v),np.std(v)
    print(f"\n{ds.upper()} (LOO, seeds {seeds}, fp={fp}): "+" ".join(f"{k} {ag(k)[0]:.2f}±{ag(k)[1]:.2f}" for k in ['HR@1','HR@5','HR@10','HR@20','MRR']))
    for seg in ('repeat','explore'):
        rr=[r['repeat_explore'][seg]['HR@10']*100 for r in recs if r.get('repeat_explore') and r['repeat_explore'][seg].get('n')]
        if rr: print(f"   {seg:8s}(n={recs[0]['repeat_explore'][seg]['n']}) HR@10={np.mean(rr):.2f}±{np.std(rr):.2f}")


NYC (LOO, seeds [42, 43, 44], fp=73c596f2920e5266): HR@1 25.89±0.16 HR@5 50.48±0.28 HR@10 57.85±0.70 HR@20 62.01±0.46 MRR 36.97±0.19
   repeat  (n=669) HR@10=88.74±1.25
   explore (n=375) HR@10=2.76±0.82

TKY (LOO, seeds [42, 43, 44], fp=f513cb5fcbe0832e): HR@1 23.64±0.13 HR@5 48.17±0.34 HR@10 57.82±0.37 HR@20 65.98±0.37 MRR 35.09±0.13
   repeat  (n=1561) HR@10=81.36±0.59
   explore (n=719) HR@10=6.72±0.26

CA (LOO, seeds [42, 43, 44], fp=3b6a753dab84043c): HR@1 17.59±0.54 HR@5 37.54±0.41 HR@10 45.95±0.59 HR@20 54.26±0.61 MRR 27.17±0.19
   repeat  (n=1767) HR@10=77.70±0.91
   explore (n=2189) HR@10=20.31±0.37


### 7. Download

In [20]:
# # 7) Download (also on Drive)
# import shutil, os
# shutil.make_archive('/content/castpoi_loo_results','zip',os.environ['CASTPOI_RESULTS'])
# from google.colab import files; files.download('/content/castpoi_loo_results.zip')

## 7) Zero-parameter revisit heuristic (LOO) — the revisit-dominance anchor

In [21]:
%%writefile heuristic_loo.py
#!/usr/bin/env python3
"""Zero-parameter revisit heuristic under the canonical LOO split + full-vocab ranking.

Score(u, c) = (#times c appears in u's history) x (global POI popularity).
Unvisited POIs get score 0 (so the heuristic can only ever rank a user's own past
POIs; explore targets fall to the bottom). Ranks use the mid-rank tie convention,
matching the learned-model evaluation. Output is directly comparable to CaST-POI /
baselines because it runs on the SAME data_loo and reports the SAME metrics, and
saves ranks.npy + check_ins_id.npy so it can enter the paired significance test.

  OFFICIAL_DIR=data_loo python heuristic_loo.py --dataset all
"""
import argparse, json, os, sys
from pathlib import Path
import numpy as np
import torch

HERE = Path(__file__).resolve().parent
RESULTS = Path(os.environ.get("CASTPOI_RESULTS", HERE / "results"))
OFFICIAL_DIR = Path(os.environ.get("OFFICIAL_DIR", HERE / "data_loo"))

from castpoi.config import resolve_config          # noqa: E402
from castpoi.official import load_official          # noqa: E402
from castpoi.data import create_dataloaders         # noqa: E402

DATASETS = ("nyc", "tky", "ca")
EVAL_KS = [1, 5, 10, 20]


def run(dataset):
    cfg = resolve_config(dataset, {})
    cfg["eval_batch_size"] = 256
    data = load_official(dataset, OFFICIAL_DIR)
    num_pois = data["num_pois"]
    _, _, tel = create_dataloaders(data, cfg)

    # pass 1: global popularity from the eval histories (train+val), pad excluded
    pop = np.zeros(num_pois, dtype=np.float64)
    for b in tel:
        src = b.get("repeat_hist", b["poi_ids"]).numpy()
        for row in src:
            np.add.at(pop, row, 1.0)
    pop[0] = 0.0
    pop = pop / max(pop.sum(), 1.0)

    # pass 2: rank each target under count(hist)*popularity, mid-rank ties
    ranks, is_rep, cids = [], [], []
    cid_all = tel.dataset.check_ins_ids
    cid_all = cid_all() if callable(cid_all) else cid_all
    ptr = 0
    for b in tel:
        src = b.get("repeat_hist", b["poi_ids"]).numpy()
        tgt = b["target_poi"].numpy()
        bs = src.shape[0]
        for i in range(bs):
            cnt = np.zeros(num_pois, dtype=np.float64)
            np.add.at(cnt, src[i], 1.0)
            cnt[0] = 0.0
            score = cnt * pop
            t = int(tgt[i])
            st = score[t]
            greater = int((score[1:] > st).sum())          # exclude pad idx 0
            equal = int((score[1:] == st).sum())            # includes target itself
            rank = greater + (equal + 1) / 2.0
            ranks.append(rank)
            is_rep.append(cnt[t] > 0)
            cids.append(int(cid_all[ptr + i]))
        ptr += bs

    ranks = np.array(ranks); is_rep = np.array(is_rep, dtype=bool)
    cids = np.array(cids, dtype=np.int64)

    def sub(mask):
        r = ranks[mask]
        if not len(r): return {"n": 0}
        return {**{f"HR@{k}": float((r <= k).mean()) for k in EVAL_KS},
                "MRR": float((1.0 / r).mean()), "n": int(len(r))}

    metrics = {**{f"HR@{k}": float((ranks <= k).mean()) for k in EVAL_KS},
               "MRR": float((1.0 / ranks).mean()), "n": int(len(ranks))}
    rep_expl = {"repeat": sub(is_rep), "explore": sub(~is_rep),
                "repeat_frac": float(is_rep.mean())}

    out = RESULTS / "runs" / "heuristic_loo" / dataset
    out.mkdir(parents=True, exist_ok=True)
    np.save(out / "ranks.npy", ranks); np.save(out / "check_ins_id.npy", cids)
    np.save(out / "is_repeat.npy", is_rep)
    rec = {"dataset": dataset, "model": "revisit_heuristic", "split": "loo",
           "data_fingerprint": data["stats"].get("data_fingerprint"),
           "test": metrics, "repeat_explore": rep_expl}
    json.dump(rec, open(out / "metrics.json", "w"), indent=2)
    print(f"[{dataset}] revisit heuristic: " +
          " ".join(f"{k} {metrics[k]*100:.2f}" for k in ("HR@1", "HR@5", "HR@10", "HR@20", "MRR")) +
          f" | repeat_frac={rep_expl['repeat_frac']*100:.1f}% | fp={rec['data_fingerprint']}")
    return rec


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dataset", default="all")
    args = ap.parse_args()
    assert OFFICIAL_DIR.exists(), f"OFFICIAL_DIR={OFFICIAL_DIR} not found (build data_loo first)."
    for ds in (list(DATASETS) if args.dataset == "all" else [args.dataset]):
        run(ds)


if __name__ == "__main__":
    main()


Writing heuristic_loo.py


In [22]:
!python heuristic_loo.py --dataset all


[official] NYC (Foursquare New York City, Foursquare)
[official] provenance: GETNext pre-split NYC_{train,val,test}.csv
[official] train:  99,666 check-ins  (11.1 MB)
[official] val  :   1,043 check-ins  (0.1 MB)
[official] test :   1,044 check-ins  (0.1 MB)
[official] note: 6 vocabulary slots never appear in any split
[official] users=1,047 POIs=4,980 check-ins=101,753 trajectories=13,938
[official] local time OK (trough 04:00, day/night 14.4) via America/New_York, column read as local
[official] fingerprint 73c596f2920e5266
[data] samples: train=98,619 val=1,043 test=1,044
[nyc] revisit heuristic: HR@1 14.75 HR@5 41.19 HR@10 54.60 HR@20 60.82 MRR 26.94 | repeat_frac=64.1% | fp=73c596f2920e5266

[official] TKY (Foursquare Tokyo, Foursquare)
[official] provenance: STHGCN filter(9,9) + global chronological 80/10/10
[official] train: 398,586 check-ins  (42.7 MB)
[official] val  :   2,280 check-ins  (0.2 MB)
[official] test :   2,280 check-ins  (0.2 MB)
[official] users=2,281 POIs=7,832 

## 8) Paired significance test (HR@10) — is the NYC tie real or noise?
Point --a at CaST-POI runs and --b at a baseline run dir (needs its ranks.npy + check_ins_id.npy).

In [23]:
%%writefile paired_sig_loo.py
#!/usr/bin/env python3
"""Per-sample paired significance test on HR@k between two models under LOO.

Aligns two models sample-by-sample via the official check_ins_id (the join key both
sides save), averages the HR@k hit indicator across seeds per sample, then does a
paired bootstrap over test samples -> HR@k delta, 95% CI, two-sided p-value. Also
reports McNemar-style discordant counts. Reads only ranks.npy + check_ins_id.npy;
writes nothing. Use it to say whether the NYC tie (Delta HR@10 = -0.10) is real or noise.

  python paired_sig_loo.py \
    --a results/runs/castpoi.model_loo_lr0.001/nyc/full \
    --b <baseline_runs>/loo/nyc/core_recbole \
    --k 10
"""
import argparse, glob, os
import numpy as np


def load_side(path, k):
    """Return dict check_ins_id -> mean hit@k across all seeds found under path."""
    rank_files = sorted(glob.glob(os.path.join(path, "**", "ranks.npy"), recursive=True))
    if not rank_files and os.path.exists(os.path.join(path, "ranks.npy")):
        rank_files = [os.path.join(path, "ranks.npy")]
    if not rank_files:
        raise FileNotFoundError(f"no ranks.npy under {path}")
    per_cid = {}
    for rf in rank_files:
        cf = os.path.join(os.path.dirname(rf), "check_ins_id.npy")
        if not os.path.exists(cf):
            raise FileNotFoundError(f"missing check_ins_id.npy next to {rf}")
        ranks = np.load(rf); cids = np.load(cf)
        hit = (ranks <= k).astype(np.float64)
        for c, h in zip(cids.tolist(), hit.tolist()):
            per_cid.setdefault(c, []).append(h)
    return {c: float(np.mean(v)) for c, v in per_cid.items()}, len(rank_files)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--a", required=True, help="run dir for model A (e.g. CaST-POI)")
    ap.add_argument("--b", required=True, help="run dir for model B (baseline)")
    ap.add_argument("--k", type=int, default=10)
    ap.add_argument("--nboot", type=int, default=20000)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--labels", nargs=2, default=["A", "B"])
    args = ap.parse_args()

    A, na = load_side(args.a, args.k)
    B, nb = load_side(args.b, args.k)
    common = sorted(set(A) & set(B))
    if not common:
        raise SystemExit("no overlapping check_ins_id between the two runs")
    la, lb = args.labels
    a = np.array([A[c] for c in common]); b = np.array([B[c] for c in common])
    n = len(common)
    hr_a, hr_b = a.mean() * 100, b.mean() * 100
    delta = hr_a - hr_b

    rng = np.random.default_rng(args.seed)
    idx = rng.integers(0, n, size=(args.nboot, n))
    boot = (a[idx].mean(1) - b[idx].mean(1)) * 100
    lo, hi = np.percentile(boot, [2.5, 97.5])
    p = 2.0 * min((boot <= 0).mean(), (boot >= 0).mean())

    # discordant counts (seed-averaged hit >= 0.5 as the indicator)
    ah, bh = a >= 0.5, b >= 0.5
    bc = int((ah & ~bh).sum()); cc = int((~ah & bh).sum())

    print(f"paired HR@{args.k} test on {n} aligned samples "
          f"({la}: {na} seed-run(s), {lb}: {nb} seed-run(s))")
    print(f"  {la} HR@{args.k} = {hr_a:.2f}   {lb} HR@{args.k} = {hr_b:.2f}")
    print(f"  Delta = {delta:+.2f}   95% CI [{lo:+.2f}, {hi:+.2f}]   p(two-sided) = {p:.4f}")
    print(f"  verdict: {'significant' if p < 0.05 else 'NOT significant (statistical tie)'}")
    print(f"  discordant: {la}-only hits = {bc}, {lb}-only hits = {cc}")


if __name__ == "__main__":
    main()


Writing paired_sig_loo.py


In [24]:
# example: CaST-POI vs the best NYC baseline (edit --b to your baseline runs path)
RES=os.environ["CASTPOI_RESULTS"]
RUNS=os.environ.get("RUNS", f"{DRIVE_ROOT}/runs")   # where recbole_baselines_loo.ipynb wrote the baselines
import glob
for ds,bestbase in [("nyc","core_recbole"),("tky","sasrec_recbole"),("ca","sasrec_recbole")]:
    a=f"{RES}/runs/castpoi.model_loo_lr0.001/{ds}/full"
    bcands=glob.glob(f"{RUNS}/loo/{ds}/{bestbase}")
    if bcands:
        print(f"===== {ds} : CaST-POI vs {bestbase} =====")
        !python paired_sig_loo.py --a "{a}" --b "{bcands[0]}" --k 10 --labels CaST-POI {bestbase}
    else:
        print(f"[{ds}] baseline runs not found at {RUNS}/loo/{ds}/{bestbase} — export them first")

[nyc] baseline runs not found at /content/drive/MyDrive/castpoi/runs/loo/nyc/core_recbole — export them first
[tky] baseline runs not found at /content/drive/MyDrive/castpoi/runs/loo/tky/sasrec_recbole — export them first
[ca] baseline runs not found at /content/drive/MyDrive/castpoi/runs/loo/ca/sasrec_recbole — export them first


In [25]:
!find /content/drive/MyDrive/castpoi -maxdepth 4 -type d -name "*recbole*" | head -40